# 18sB — Expanded sample aggregation and freeze

This second stage reads only the canonical outputs produced by
18sA. It no longer contains legacy source-schema inference.

It constructs the 4,532-row candidate grid, the 3,889-row exact
common-support panel and the 350 complete event books, then writes
the definitive 18s expanded sample release.

In [1]:
from __future__ import annotations

import hashlib
import json
import math
import platform
import sys
from datetime import datetime, timezone
from pathlib import Path
from zoneinfo import ZoneInfo

import numpy as np
import pandas as pd
from IPython.display import display

ROOT = Path.cwd().resolve()
if not (ROOT / ".git").exists():
    raise RuntimeError(
        f"Run from repository root, not {ROOT}"
    )

UTC = timezone.utc
HKT = ZoneInfo("Asia/Hong_Kong")
EPSILON = 1e-6

RULES = [
    "24h_prior",
    "12h_prior",
    "6h_prior",
    "event_day_open",
]
RULE_ORDER = {
    rule: index
    for index, rule in enumerate(RULES)
}
RULE_OFFSET_HOURS = {
    "24h_prior": -24,
    "12h_prior": -12,
    "6h_prior": -6,
    "event_day_open": 0,
}

EXPECTED = {
    "target_rows": 1133,
    "dates": 103,
    "candidate_rows": 4532,
    "market_rows": 4186,
    "weather_rows": 4125,
    "common_rows": 3889,
    "common_groups": 355,
    "complete_books": 350,
    "complete_contract_rows": 3850,
    "exclusion_rows": 643,
}

EXPECTED_RULE = {
    "24h_prior": {
        "candidate": 1133,
        "market": 976,
        "weather": 1056,
        "common": 921,
        "groups": 85,
        "books": 82,
    },
    "12h_prior": {
        "candidate": 1133,
        "market": 1055,
        "weather": 1023,
        "common": 978,
        "groups": 89,
        "books": 88,
    },
    "6h_prior": {
        "candidate": 1133,
        "market": 1077,
        "weather": 1001,
        "common": 967,
        "groups": 88,
        "books": 87,
    },
    "event_day_open": {
        "candidate": 1133,
        "market": 1078,
        "weather": 1045,
        "common": 1023,
        "groups": 93,
        "books": 93,
    },
}

ADAPTER_DIR = (
    ROOT
    / "data/processed/18sA_canonical_source_adapters"
)
ADAPTER_TARGET = (
    ADAPTER_DIR / "18sA_canonical_contract_outcome_panel.csv"
)
ADAPTER_MARKET = (
    ADAPTER_DIR / "18sA_canonical_market_panel.csv"
)
ADAPTER_WEATHER = (
    ADAPTER_DIR / "18sA_canonical_weather_panel.csv"
)
ADAPTER_SOURCE_INVENTORY = (
    ADAPTER_DIR / "18sA_source_inventory.csv"
)
ADAPTER_SCHEMA = (
    ADAPTER_DIR / "18sA_schema_resolution.json"
)
ADAPTER_SUMMARY = (
    ADAPTER_DIR / "18sA_summary.json"
)
ADAPTER_MANIFEST = (
    ADAPTER_DIR / "18sA_sha256_manifest.csv"
)

OUT = (
    ROOT
    / "data/processed/18s_expanded_march_june_canonical_sample"
)
REPORT = (
    ROOT
    / "reports/18s_expanded_march_june_canonical_sample"
)
OUT.mkdir(parents=True, exist_ok=True)
REPORT.mkdir(parents=True, exist_ok=True)

required_adapter_files = [
    ADAPTER_TARGET,
    ADAPTER_MARKET,
    ADAPTER_WEATHER,
    ADAPTER_SOURCE_INVENTORY,
    ADAPTER_SCHEMA,
    ADAPTER_SUMMARY,
    ADAPTER_MANIFEST,
]

for path in required_adapter_files:
    if not path.is_file():
        raise FileNotFoundError(
            f"Required 18sA adapter output is missing: {path}"
        )


def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as handle:
        for chunk in iter(
            lambda: handle.read(1024 * 1024),
            b"",
        ):
            digest.update(chunk)
    return digest.hexdigest()


def parse_bool(
    series: pd.Series,
    *,
    name: str,
) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype(bool)

    parsed = (
        series.astype(str)
        .str.strip()
        .str.lower()
        .map(
            {
                "true": True,
                "false": False,
                "1": True,
                "0": False,
                "yes": True,
                "no": False,
            }
        )
    )

    if parsed.isna().any():
        raise ValueError(
            f"Could not parse Boolean column {name}: "
            f"{series.loc[parsed.isna()].drop_duplicates().tolist()}"
        )

    return parsed.astype(bool)


def decision_cutoff_hkt(
    event_date: pd.Timestamp,
    decision_rule: str,
) -> pd.Timestamp:
    local_open = pd.Timestamp(
        year=event_date.year,
        month=event_date.month,
        day=event_date.day,
        hour=0,
        tz=HKT,
    )
    return local_open + pd.Timedelta(
        hours=RULE_OFFSET_HOURS[decision_rule]
    )


with ADAPTER_SUMMARY.open(encoding="utf-8") as handle:
    adapter_summary = json.load(handle)

if adapter_summary.get("verdict") != "PASS":
    raise AssertionError(
        "18sA is not a PASS release: "
        f"{adapter_summary.get('verdict')}"
    )

adapter_manifest = pd.read_csv(ADAPTER_MANIFEST)
adapter_manifest_failures = []

for row in adapter_manifest.itertuples(index=False):
    path = ROOT / row.path

    if not path.is_file():
        adapter_manifest_failures.append(
            f"MISSING: {row.path}"
        )
        continue

    digest = sha256_file(path)

    if digest != row.sha256:
        adapter_manifest_failures.append(
            f"HASH: {row.path}"
        )

    if path.stat().st_size != int(row.size_bytes):
        adapter_manifest_failures.append(
            f"SIZE: {row.path}"
        )

if adapter_manifest_failures:
    raise AssertionError(
        "18sA manifest verification failed:\n"
        + "\n".join(adapter_manifest_failures)
    )

target = pd.read_csv(
    ADAPTER_TARGET,
    dtype={
        "market_id": str,
        "condition_id": str,
        "event_id": str,
        "selected_yes_token_id": str,
        "no_token_id": str,
    },
    low_memory=False,
)
market = pd.read_csv(
    ADAPTER_MARKET,
    dtype={
        "market_id": str,
        "condition_id": str,
        "selected_yes_token_id": str,
    },
    low_memory=False,
)
weather = pd.read_csv(
    ADAPTER_WEATHER,
    dtype={
        "market_id": str,
        "condition_id": str,
        "selected_yes_token_id": str,
    },
    low_memory=False,
)

for frame in [target, market, weather]:
    frame["event_date"] = pd.to_datetime(
        frame["event_date"],
        errors="raise",
    )

market["decision_cutoff_utc"] = pd.to_datetime(
    market["decision_cutoff_utc"],
    utc=True,
    errors="raise",
)
market["selected_price_timestamp_utc"] = pd.to_datetime(
    market["selected_price_timestamp_utc"],
    utc=True,
    errors="raise",
)

weather["decision_cutoff_utc"] = pd.to_datetime(
    weather["decision_cutoff_utc"],
    utc=True,
    errors="raise",
)
weather["selected_run_initialisation_utc"] = pd.to_datetime(
    weather["selected_run_initialisation_utc"],
    utc=True,
    errors="raise",
)
weather["selected_run_available_utc"] = pd.to_datetime(
    weather["selected_run_available_utc"],
    utc=True,
    errors="raise",
)

for frame_name, frame, expected_rows in [
    ("target", target, EXPECTED["target_rows"]),
    ("market", market, EXPECTED["market_rows"]),
    ("weather", weather, EXPECTED["weather_rows"]),
]:
    if len(frame) != expected_rows:
        raise AssertionError(
            f"{frame_name}: expected {expected_rows}, "
            f"found {len(frame)}"
        )

if target["event_date"].nunique() != EXPECTED["dates"]:
    raise AssertionError(
        "Adapter target does not contain 103 dates"
    )

if target.duplicated(
    ["event_date", "market_id"]
).any():
    raise AssertionError(
        "Duplicate target date-market keys"
    )

if not target.groupby(
    "event_date"
)["market_id"].size().eq(11).all():
    raise AssertionError(
        "A target date does not contain 11 contracts"
    )

if not target.groupby(
    "event_date"
)["Y_event_int"].sum().eq(1).all():
    raise AssertionError(
        "A target date does not contain exactly one winner"
    )

exact_key = [
    "event_date",
    "market_id",
    "decision_rule",
]

for frame_name, frame in [
    ("market", market),
    ("weather", weather),
]:
    if frame.duplicated(exact_key).any():
        raise AssertionError(
            f"Duplicate exact keys in {frame_name}"
        )

if not (
    market["selected_price_timestamp_utc"]
    <= market["decision_cutoff_utc"]
).all():
    raise AssertionError(
        "A market timestamp is after its cut-off"
    )

if not (
    weather["selected_run_available_utc"]
    <= weather["decision_cutoff_utc"]
).all():
    raise AssertionError(
        "A weather run was unavailable at its cut-off"
    )

print("18sA adapter input verification: PASS")
print(f"Canonical targets: {len(target):,}")
print(f"Canonical market rows: {len(market):,}")
print(f"Canonical weather rows: {len(weather):,}")

# ------------------------------------------------------------------
# Expanded candidate and exact-support panels.
# ------------------------------------------------------------------

rules = pd.DataFrame(
    {
        "decision_rule": RULES,
        "decision_rule_order": range(len(RULES)),
    }
)

candidate = (
    target.assign(_cross_key=1)
    .merge(
        rules.assign(_cross_key=1),
        on="_cross_key",
        how="inner",
    )
    .drop(columns="_cross_key")
)

candidate["decision_cutoff_hkt"] = [
    decision_cutoff_hkt(date, rule)
    for date, rule in zip(
        candidate["event_date"],
        candidate["decision_rule"],
    )
]
candidate["decision_cutoff_utc"] = candidate[
    "decision_cutoff_hkt"
].map(lambda value: value.tz_convert("UTC"))

market_ready_keys = (
    market[exact_key]
    .drop_duplicates()
    .assign(market_price_ready=True)
)
weather_ready_keys = (
    weather[exact_key]
    .drop_duplicates()
    .assign(weather_path_ready=True)
)

candidate = (
    candidate.merge(
        market_ready_keys,
        on=exact_key,
        how="left",
        validate="one_to_one",
    )
    .merge(
        weather_ready_keys,
        on=exact_key,
        how="left",
        validate="one_to_one",
    )
)

candidate["market_price_ready"] = candidate[
    "market_price_ready"
].fillna(False).astype(bool)
candidate["weather_path_ready"] = candidate[
    "weather_path_ready"
].fillna(False).astype(bool)
candidate["common_support"] = (
    candidate["market_price_ready"]
    & candidate["weather_path_ready"]
)

candidate["support_exclusion_reason"] = np.select(
    [
        (
            ~candidate["market_price_ready"]
            & candidate["weather_path_ready"]
        ),
        (
            candidate["market_price_ready"]
            & ~candidate["weather_path_ready"]
        ),
        (
            ~candidate["market_price_ready"]
            & ~candidate["weather_path_ready"]
        ),
    ],
    [
        "MARKET_PRICE_MISSING",
        "WEATHER_PATH_NOT_READY",
        "MARKET_PRICE_MISSING;WEATHER_PATH_NOT_READY",
    ],
    default="",
)

if len(candidate) != EXPECTED["candidate_rows"]:
    raise AssertionError(
        f"Expected 4,532 candidate rows, found {len(candidate)}"
    )

if int(candidate["market_price_ready"].sum()) != EXPECTED[
    "market_rows"
]:
    raise AssertionError(
        "Candidate market-ready count is incorrect"
    )

if int(candidate["weather_path_ready"].sum()) != EXPECTED[
    "weather_rows"
]:
    raise AssertionError(
        "Candidate weather-ready count is incorrect"
    )

if int(candidate["common_support"].sum()) != EXPECTED[
    "common_rows"
]:
    raise AssertionError(
        "Candidate common-support count is incorrect"
    )

exclusions = candidate.loc[
    ~candidate["common_support"]
].copy()

expected_exclusions = {
    "MARKET_PRICE_MISSING": 236,
    "WEATHER_PATH_NOT_READY": 297,
    "MARKET_PRICE_MISSING;WEATHER_PATH_NOT_READY": 110,
}
actual_exclusions = (
    exclusions["support_exclusion_reason"]
    .value_counts()
    .to_dict()
)

if actual_exclusions != expected_exclusions:
    raise AssertionError(
        "Expanded exclusion counts differ:\n"
        f"Expected={expected_exclusions}\n"
        f"Actual={actual_exclusions}"
    )

metadata_columns = [
    "event_date",
    "market_id",
    "condition_id",
    "event_id",
    "event_slug",
    "market_slug",
    "question",
    "canonical_label",
    "event_type",
    "contract_event_type",
    "lower_bound_c",
    "upper_bound_c",
    "bound_reference_c",
    "bound_reconstruction_applied",
    "bound_reference_source",
    "source_lower_bound_column",
    "source_upper_bound_column",
    "selected_yes_token_id",
    "no_token_id",
    "hko_daily_max_c",
    "Y_event_int",
    "Y_no_int",
    "sample_block",
    "final_modelling_split",
    "outcome_history_admissibility_status",
]

market_columns = exact_key + [
    "p_market",
    "selected_price_timestamp_utc",
    "price_staleness_hours",
    "market_binary_brier",
    "market_binary_log_score",
    "source_price_timestamp_column",
    "source_cutoff_column",
]

weather_columns = exact_key + [
    "selected_run_initialisation_utc",
    "selected_run_available_utc",
    "selected_run_key",
    "forecast_daily_max_c",
    "forecast_error_c",
    "absolute_error_c",
    "deterministic_forecast_event_indicator",
    "source_forecast_max_column",
    "source_run_initialisation_column",
    "source_run_available_column",
    "run_metadata_reconstructed",
]

common = (
    candidate.loc[
        candidate["common_support"],
        metadata_columns
        + [
            "decision_rule",
            "decision_rule_order",
            "decision_cutoff_hkt",
            "decision_cutoff_utc",
        ],
    ]
    .merge(
        market[market_columns],
        on=exact_key,
        how="left",
        validate="one_to_one",
    )
    .merge(
        weather[weather_columns],
        on=exact_key,
        how="left",
        validate="one_to_one",
    )
)

if len(common) != EXPECTED["common_rows"]:
    raise AssertionError(
        f"Expected 3,889 common rows, found {len(common)}"
    )

if common[
    [
        "p_market",
        "selected_price_timestamp_utc",
        "forecast_daily_max_c",
        "selected_run_available_utc",
    ]
].isna().any().any():
    raise AssertionError(
        "A common-support row lacks required market or weather data"
    )

if not (
    common["selected_price_timestamp_utc"]
    <= common["decision_cutoff_utc"]
).all():
    raise AssertionError(
        "A common-support market timestamp is after its cut-off"
    )

if not (
    common["selected_run_available_utc"]
    <= common["decision_cutoff_utc"]
).all():
    raise AssertionError(
        "A common-support weather run is unavailable at its cut-off"
    )

book_key = ["event_date", "decision_rule"]
common["common_book_size"] = common.groupby(
    book_key
)["market_id"].transform("size")
common["complete_common_book"] = common[
    "common_book_size"
].eq(11)

common_group_sizes = common.groupby(
    book_key
)["market_id"].size()

if len(common_group_sizes) != EXPECTED["common_groups"]:
    raise AssertionError(
        "Expected 355 common-support date-rule groups"
    )

if int(common_group_sizes.eq(11).sum()) != EXPECTED[
    "complete_books"
]:
    raise AssertionError(
        "Expected 350 complete common-support books"
    )

complete_contract = common.loc[
    common["complete_common_book"]
].copy()

if len(complete_contract) != EXPECTED[
    "complete_contract_rows"
]:
    raise AssertionError(
        "Expected 3,850 complete-book contract rows"
    )

complete_contract[
    "market_book_probability_sum"
] = complete_contract.groupby(
    book_key
)["p_market"].transform("sum")
complete_contract[
    "p_market_normalised"
] = (
    complete_contract["p_market"]
    / complete_contract["market_book_probability_sum"]
)

if not np.isclose(
    complete_contract.groupby(
        book_key
    )["p_market_normalised"].sum(),
    1.0,
    rtol=0.0,
    atol=1e-12,
).all():
    raise AssertionError(
        "Normalised complete-book probabilities do not sum to one"
    )

if not complete_contract.groupby(
    book_key
)["Y_event_int"].sum().eq(1).all():
    raise AssertionError(
        "A complete book lacks exactly one realised winner"
    )

if not complete_contract.groupby(
    book_key
)[
    "deterministic_forecast_event_indicator"
].sum().eq(1).all():
    raise AssertionError(
        "A complete book lacks exactly one deterministic selection"
    )

# ------------------------------------------------------------------
# Tie-aware complete-book panel.
# ------------------------------------------------------------------

book_rows = []

for (event_date, decision_rule), book in (
    complete_contract.groupby(
        book_key,
        sort=True,
    )
):
    actual = book.loc[
        book["Y_event_int"].eq(1)
    ]
    deterministic = book.loc[
        book[
            "deterministic_forecast_event_indicator"
        ].eq(1)
    ]

    if len(actual) != 1 or len(deterministic) != 1:
        raise AssertionError(
            "Unexpected winner or deterministic-selection count"
        )

    actual = actual.iloc[0]
    deterministic = deterministic.iloc[0]

    modal_probability = float(book["p_market"].max())
    modal_rows = book.loc[
        np.isclose(
            book["p_market"],
            modal_probability,
            rtol=0.0,
            atol=1e-12,
        )
    ].sort_values("market_id")

    modal_ids = (
        modal_rows["market_id"]
        .astype(str)
        .tolist()
    )
    modal_labels = (
        modal_rows["canonical_label"]
        .astype(str)
        .tolist()
    )

    winning_probability_raw = float(
        actual["p_market"]
    )
    winning_probability_normalised = float(
        actual["p_market_normalised"]
    )

    book_rows.append(
        {
            "event_date": event_date,
            "sample_block": actual["sample_block"],
            "decision_rule": decision_rule,
            "decision_rule_order": RULE_ORDER[
                decision_rule
            ],
            "decision_cutoff_utc": actual[
                "decision_cutoff_utc"
            ],
            "n_contracts": len(book),
            "market_book_probability_sum": float(
                book["p_market"].sum()
            ),
            "market_book_probability_error_vs_one": float(
                book["p_market"].sum() - 1.0
            ),
            "hko_daily_max_c": float(
                actual["hko_daily_max_c"]
            ),
            "forecast_daily_max_c": float(
                actual["forecast_daily_max_c"]
            ),
            "forecast_error_c": float(
                actual["forecast_error_c"]
            ),
            "absolute_error_c": float(
                actual["absolute_error_c"]
            ),
            "actual_winning_market_id": actual[
                "market_id"
            ],
            "actual_winning_label": actual[
                "canonical_label"
            ],
            "actual_winner_probability_raw": (
                winning_probability_raw
            ),
            "actual_winner_probability_normalised": (
                winning_probability_normalised
            ),
            "deterministic_selected_market_id": (
                deterministic["market_id"]
            ),
            "deterministic_selected_label": (
                deterministic["canonical_label"]
            ),
            "deterministic_exact_contract_hit": int(
                deterministic["Y_event_int"] == 1
            ),
            "n_market_modal_contracts": len(
                modal_rows
            ),
            "market_modal_tie": len(
                modal_rows
            )
            > 1,
            "market_modal_market_ids": "|".join(
                modal_ids
            ),
            "market_modal_labels": "|".join(
                modal_labels
            ),
            "market_modal_probability_raw": (
                modal_probability
            ),
            "market_modal_contains_actual_winner": int(
                str(actual["market_id"]) in modal_ids
            ),
            "market_modal_contains_deterministic": int(
                str(deterministic["market_id"])
                in modal_ids
            ),
            "raw_categorical_log_score": -math.log(
                min(
                    max(
                        winning_probability_raw,
                        EPSILON,
                    ),
                    1.0 - EPSILON,
                )
            ),
            "normalised_categorical_log_score": -math.log(
                min(
                    max(
                        winning_probability_normalised,
                        EPSILON,
                    ),
                    1.0 - EPSILON,
                )
            ),
            "raw_multiclass_brier": float(
                np.square(
                    book["p_market"].to_numpy(
                        dtype=float
                    )
                    - book["Y_event_int"].to_numpy(
                        dtype=float
                    )
                ).sum()
            ),
            "normalised_multiclass_brier": float(
                np.square(
                    book[
                        "p_market_normalised"
                    ].to_numpy(dtype=float)
                    - book["Y_event_int"].to_numpy(
                        dtype=float
                    )
                ).sum()
            ),
        }
    )

books = pd.DataFrame(book_rows).sort_values(
    ["event_date", "decision_rule_order"]
).reset_index(drop=True)

if len(books) != EXPECTED["complete_books"]:
    raise AssertionError(
        f"Expected 350 book rows, found {len(books)}"
    )

# ------------------------------------------------------------------
# Flow, date inventory and diagnostics.
# ------------------------------------------------------------------

flow_rows = []

for sample_block in [
    "march_may_baseline",
    "june_external_extension",
    "ALL",
]:
    candidate_block = (
        candidate
        if sample_block == "ALL"
        else candidate.loc[
            candidate["sample_block"].eq(
                sample_block
            )
        ]
    )
    common_block = (
        common
        if sample_block == "ALL"
        else common.loc[
            common["sample_block"].eq(
                sample_block
            )
        ]
    )
    books_block = (
        books
        if sample_block == "ALL"
        else books.loc[
            books["sample_block"].eq(
                sample_block
            )
        ]
    )

    for rule in RULES:
        candidate_rule = candidate_block.loc[
            candidate_block["decision_rule"].eq(rule)
        ]
        common_rule = common_block.loc[
            common_block["decision_rule"].eq(rule)
        ]
        books_rule = books_block.loc[
            books_block["decision_rule"].eq(rule)
        ]

        flow_rows.append(
            {
                "sample_block": sample_block,
                "decision_rule": rule,
                "decision_rule_order": RULE_ORDER[
                    rule
                ],
                "candidate_contract_rows": len(
                    candidate_rule
                ),
                "market_price_ready_rows": int(
                    candidate_rule[
                        "market_price_ready"
                    ].sum()
                ),
                "weather_path_ready_rows": int(
                    candidate_rule[
                        "weather_path_ready"
                    ].sum()
                ),
                "common_support_contract_rows": int(
                    candidate_rule[
                        "common_support"
                    ].sum()
                ),
                "support_exclusion_rows": int(
                    (
                        ~candidate_rule[
                            "common_support"
                        ]
                    ).sum()
                ),
                "common_support_groups": int(
                    common_rule[
                        "event_date"
                    ].nunique()
                ),
                "complete_common_books": len(
                    books_rule
                ),
                "incomplete_common_groups": int(
                    common_rule.groupby(
                        "event_date"
                    )["market_id"]
                    .size()
                    .lt(11)
                    .sum()
                ),
            }
        )

flow = pd.DataFrame(flow_rows)
all_flow = flow.loc[
    flow["sample_block"].eq("ALL")
].set_index("decision_rule")

for rule, expected in EXPECTED_RULE.items():
    actual = {
        "candidate": int(
            all_flow.loc[
                rule,
                "candidate_contract_rows",
            ]
        ),
        "market": int(
            all_flow.loc[
                rule,
                "market_price_ready_rows",
            ]
        ),
        "weather": int(
            all_flow.loc[
                rule,
                "weather_path_ready_rows",
            ]
        ),
        "common": int(
            all_flow.loc[
                rule,
                "common_support_contract_rows",
            ]
        ),
        "groups": int(
            all_flow.loc[
                rule,
                "common_support_groups",
            ]
        ),
        "books": int(
            all_flow.loc[
                rule,
                "complete_common_books",
            ]
        ),
    }

    if actual != expected:
        raise AssertionError(
            f"Rule-level totals differ for {rule}:\n"
            f"Expected={expected}\n"
            f"Actual={actual}"
        )

date_inventory = (
    target.groupby(
        ["event_date", "sample_block"],
        as_index=False,
    )
    .agg(
        certified_contracts=("market_id", "size"),
        realised_winners=("Y_event_int", "sum"),
        hko_daily_max_c=(
            "hko_daily_max_c",
            "first",
        ),
    )
)

for rule in RULES:
    rule_candidate = candidate.loc[
        candidate["decision_rule"].eq(rule)
    ]

    counts = (
        rule_candidate.groupby("event_date")
        .agg(
            market_ready=(
                "market_price_ready",
                "sum",
            ),
            weather_ready=(
                "weather_path_ready",
                "sum",
            ),
            common_rows=(
                "common_support",
                "sum",
            ),
        )
        .rename(
            columns={
                "market_ready": (
                    f"{rule}_market_ready_rows"
                ),
                "weather_ready": (
                    f"{rule}_weather_ready_rows"
                ),
                "common_rows": (
                    f"{rule}_common_rows"
                ),
            }
        )
        .reset_index()
    )

    date_inventory = date_inventory.merge(
        counts,
        on="event_date",
        how="left",
        validate="one_to_one",
    )

    complete_dates = set(
        books.loc[
            books["decision_rule"].eq(rule),
            "event_date",
        ]
    )
    date_inventory[
        f"{rule}_complete_common_book"
    ] = date_inventory["event_date"].isin(
        complete_dates
    )

date_inventory["any_exact_common_support"] = (
    date_inventory[
        [
            f"{rule}_common_rows"
            for rule in RULES
        ]
    ].sum(axis=1)
    > 0
)
date_inventory["any_complete_common_book"] = (
    date_inventory[
        [
            f"{rule}_complete_common_book"
            for rule in RULES
        ]
    ].any(axis=1)
)
date_inventory["final_modelling_split"] = (
    "UNASSIGNED"
)
date_inventory[
    "postprocessing_history_admissibility"
] = "NOT_YET_APPLIED"

binary_rows = []
categorical_rows = []
weather_rows = []

for sample_block in [
    "march_may_baseline",
    "june_external_extension",
    "ALL",
]:
    common_block = (
        common
        if sample_block == "ALL"
        else common.loc[
            common["sample_block"].eq(
                sample_block
            )
        ]
    )
    books_block = (
        books
        if sample_block == "ALL"
        else books.loc[
            books["sample_block"].eq(
                sample_block
            )
        ]
    )

    for rule in RULES:
        contract_subset = common_block.loc[
            common_block["decision_rule"].eq(rule)
        ]
        book_subset = books_block.loc[
            books_block["decision_rule"].eq(rule)
        ]

        if not contract_subset.empty:
            binary_rows.append(
                {
                    "sample_block": sample_block,
                    "decision_rule": rule,
                    "decision_rule_order": RULE_ORDER[
                        rule
                    ],
                    "n_contract_rows": len(
                        contract_subset
                    ),
                    "n_dates": contract_subset[
                        "event_date"
                    ].nunique(),
                    "mean_market_brier": contract_subset[
                        "market_binary_brier"
                    ].mean(),
                    "mean_market_log_score": contract_subset[
                        "market_binary_log_score"
                    ].mean(),
                    "median_market_brier": contract_subset[
                        "market_binary_brier"
                    ].median(),
                    "median_market_log_score": contract_subset[
                        "market_binary_log_score"
                    ].median(),
                    "mean_market_probability": contract_subset[
                        "p_market"
                    ].mean(),
                    "outcome_rate": contract_subset[
                        "Y_event_int"
                    ].mean(),
                }
            )

        if not book_subset.empty:
            categorical_rows.append(
                {
                    "sample_block": sample_block,
                    "decision_rule": rule,
                    "decision_rule_order": RULE_ORDER[
                        rule
                    ],
                    "n_complete_books": len(
                        book_subset
                    ),
                    "mean_book_probability_sum": book_subset[
                        "market_book_probability_sum"
                    ].mean(),
                    "mean_abs_book_probability_error": book_subset[
                        "market_book_probability_error_vs_one"
                    ].abs().mean(),
                    "mean_normalised_categorical_log_score": book_subset[
                        "normalised_categorical_log_score"
                    ].mean(),
                    "mean_normalised_multiclass_brier": book_subset[
                        "normalised_multiclass_brier"
                    ].mean(),
                    "market_modal_tied_books": int(
                        book_subset[
                            "market_modal_tie"
                        ].sum()
                    ),
                    "market_modal_contains_actual_winner_rate": book_subset[
                        "market_modal_contains_actual_winner"
                    ].mean(),
                    "deterministic_exact_contract_hit_rate": book_subset[
                        "deterministic_exact_contract_hit"
                    ].mean(),
                    "market_modal_contains_deterministic_rate": book_subset[
                        "market_modal_contains_deterministic"
                    ].mean(),
                }
            )

            weather_rows.append(
                {
                    "sample_block": sample_block,
                    "decision_rule": rule,
                    "decision_rule_order": RULE_ORDER[
                        rule
                    ],
                    "n_complete_books": len(
                        book_subset
                    ),
                    "mean_forecast_daily_max_c": book_subset[
                        "forecast_daily_max_c"
                    ].mean(),
                    "mean_hko_daily_max_c": book_subset[
                        "hko_daily_max_c"
                    ].mean(),
                    "mean_error_c": book_subset[
                        "forecast_error_c"
                    ].mean(),
                    "mae_c": book_subset[
                        "absolute_error_c"
                    ].mean(),
                    "rmse_c": math.sqrt(
                        np.square(
                            book_subset[
                                "forecast_error_c"
                            ]
                        ).mean()
                    ),
                    "underforecast_rate": (
                        book_subset[
                            "forecast_error_c"
                        ]
                        < 0
                    ).mean(),
                    "deterministic_exact_contract_hit_rate": book_subset[
                        "deterministic_exact_contract_hit"
                    ].mean(),
                }
            )

binary = pd.DataFrame(binary_rows)
categorical = pd.DataFrame(categorical_rows)
weather_summary = pd.DataFrame(weather_rows)

# ------------------------------------------------------------------
# Integrity checks.
# ------------------------------------------------------------------

check_rows = []

def add_check(
    name: str,
    passed: bool,
    detail: str,
) -> None:
    check_rows.append(
        {
            "check": name,
            "passed": bool(passed),
            "detail": detail,
            "blocking": True,
        }
    )

add_check(
    "adapter_summary_pass",
    adapter_summary.get("verdict") == "PASS",
    str(adapter_summary.get("verdict")),
)
add_check(
    "target_rows_1133",
    len(target) == 1133,
    f"rows={len(target)}",
)
add_check(
    "dates_103",
    target["event_date"].nunique() == 103,
    f"dates={target['event_date'].nunique()}",
)
add_check(
    "candidate_rows_4532",
    len(candidate) == 4532,
    f"rows={len(candidate)}",
)
add_check(
    "market_rows_4186",
    len(market) == 4186,
    f"rows={len(market)}",
)
add_check(
    "weather_rows_4125",
    len(weather) == 4125,
    f"rows={len(weather)}",
)
add_check(
    "common_rows_3889",
    len(common) == 3889,
    f"rows={len(common)}",
)
add_check(
    "common_groups_355",
    len(common_group_sizes) == 355,
    f"groups={len(common_group_sizes)}",
)
add_check(
    "complete_books_350",
    len(books) == 350,
    f"books={len(books)}",
)
add_check(
    "complete_contract_rows_3850",
    len(complete_contract) == 3850,
    f"rows={len(complete_contract)}",
)
add_check(
    "market_no_lookahead",
    (
        market["selected_price_timestamp_utc"]
        <= market["decision_cutoff_utc"]
    ).all(),
    "all market timestamps at or before cut-off",
)
add_check(
    "weather_run_admissibility",
    (
        weather["selected_run_available_utc"]
        <= weather["decision_cutoff_utc"]
    ).all(),
    "all weather runs available by cut-off",
)
add_check(
    "normalised_books_sum_to_one",
    np.isclose(
        complete_contract.groupby(
            book_key
        )["p_market_normalised"].sum(),
        1.0,
        rtol=0.0,
        atol=1e-12,
    ).all(),
    "complete books only",
)
add_check(
    "final_split_unassigned",
    target[
        "final_modelling_split"
    ].eq("UNASSIGNED").all(),
    "split assignment deferred",
)
add_check(
    "outcome_history_admissibility_pending",
    target[
        "outcome_history_admissibility_status"
    ].eq("NOT_YET_APPLIED").all(),
    "HKO publication-time filter deferred",
)
add_check(
    "no_gaussian_bridge_columns",
    not any(
        (
            "gaussian" in column.lower()
            or "ecmwf_proxy" in column.lower()
            or column.lower().startswith("sigma")
        )
        for frame in [
            target,
            market,
            weather,
            candidate,
            common,
            complete_contract,
            books,
        ]
        for column in frame.columns
    ),
    "all canonical and final panels",
)

integrity = pd.DataFrame(check_rows)

if not integrity["passed"].all():
    raise AssertionError(
        "18sB blocking checks failed:\n"
        + integrity.loc[
            ~integrity["passed"]
        ].to_string(index=False)
    )

issues = pd.DataFrame(
    columns=[
        "issue_level",
        "issue_code",
        "event_date",
        "market_id",
        "decision_rule",
        "detail",
        "blocking",
    ]
)

# ------------------------------------------------------------------
# Write final 18s outputs.
# ------------------------------------------------------------------

frames = {
    "target": target,
    "market": market,
    "weather": weather,
    "support": candidate,
    "common": common,
    "complete_contract": complete_contract,
    "books": books,
    "exclusions": exclusions,
    "flow": flow,
    "dates": date_inventory,
    "binary": binary,
    "categorical": categorical,
    "weather_summary": weather_summary,
    "integrity": integrity,
    "issues": issues,
}

paths = {
    "target": (
        OUT
        / "18s_expanded_certified_contract_outcome_panel.csv"
    ),
    "market": (
        OUT / "18s_expanded_market_scoring_panel.csv"
    ),
    "weather": (
        OUT
        / "18s_expanded_deterministic_weather_panel.csv"
    ),
    "support": (
        OUT / "18s_expanded_support_audit_panel.csv"
    ),
    "common": (
        OUT
        / "18s_expanded_exact_common_support_panel.csv"
    ),
    "complete_contract": (
        OUT
        / "18s_expanded_complete_book_contract_panel.csv"
    ),
    "books": (
        OUT / "18s_expanded_complete_book_panel.csv"
    ),
    "exclusions": (
        OUT / "18s_expanded_support_exclusions.csv"
    ),
    "flow": (
        OUT / "18s_expanded_support_flow_summary.csv"
    ),
    "dates": (
        OUT / "18s_expanded_date_inventory.csv"
    ),
    "binary": (
        OUT
        / "18s_expanded_market_binary_score_summary.csv"
    ),
    "categorical": (
        OUT
        / "18s_expanded_market_categorical_score_summary.csv"
    ),
    "weather_summary": (
        OUT / "18s_expanded_weather_error_summary.csv"
    ),
    "integrity": (
        OUT / "18s_expanded_integrity_checks.csv"
    ),
    "issues": (
        OUT / "18s_expanded_sample_issues.csv"
    ),
}

for key, frame in frames.items():
    output = frame.copy()

    if "event_date" in output.columns:
        output["event_date"] = pd.to_datetime(
            output["event_date"]
        ).dt.date.astype(str)

    for column in output.columns:
        lowered = column.lower()
        if (
            "timestamp" in lowered
            or lowered.endswith("_utc")
            or lowered.endswith("_hkt")
            or "initialisation" in lowered
            or "initialization" in lowered
            or "available" in lowered
        ):
            output[column] = output[
                column
            ].astype(str)

    output.to_csv(paths[key], index=False)

upstream_source_inventory = pd.read_csv(
    ADAPTER_SOURCE_INVENTORY
)

adapter_output_rows = []
for role, path, rows in [
    (
        "canonical_target_adapter_output",
        ADAPTER_TARGET,
        len(target),
    ),
    (
        "canonical_market_adapter_output",
        ADAPTER_MARKET,
        len(market),
    ),
    (
        "canonical_weather_adapter_output",
        ADAPTER_WEATHER,
        len(weather),
    ),
    (
        "adapter_schema_resolution",
        ADAPTER_SCHEMA,
        1,
    ),
    (
        "adapter_summary",
        ADAPTER_SUMMARY,
        1,
    ),
]:
    adapter_output_rows.append(
        {
            "input_role": role,
            "path": str(path.relative_to(ROOT)),
            "rows": rows,
            "sha256": sha256_file(path),
        }
    )

source_inventory = pd.concat(
    [
        upstream_source_inventory,
        pd.DataFrame(adapter_output_rows),
    ],
    ignore_index=True,
)

source_inventory_path = (
    OUT / "18s_expanded_source_inventory.csv"
)
source_inventory.to_csv(
    source_inventory_path,
    index=False,
)

summary = {
    "step": "18s",
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "verdict": "PASS",
    "pipeline_structure": [
        "18sA_canonical_source_adapters",
        "18sB_expanded_sample_freeze",
    ],
    "certified_contracts": int(len(target)),
    "certified_dates": int(
        target["event_date"].nunique()
    ),
    "contract_decision_candidates": int(
        len(candidate)
    ),
    "market_ready_rows": int(len(market)),
    "weather_ready_rows": int(len(weather)),
    "exact_common_support_rows": int(len(common)),
    "common_support_date_rule_groups": int(
        len(common_group_sizes)
    ),
    "complete_common_support_books": int(
        len(books)
    ),
    "complete_book_contract_rows": int(
        len(complete_contract)
    ),
    "support_exclusion_rows": int(
        len(exclusions)
    ),
    "support_exclusions_by_reason": {
        key: int(value)
        for key, value in actual_exclusions.items()
    },
    "sample_blocks": {
        "march_may_baseline": {
            "contracts": 803,
            "dates": 73,
            "common_rows": 2635,
            "complete_books": 236,
        },
        "june_external_extension": {
            "contracts": 330,
            "dates": 30,
            "common_rows": 1254,
            "complete_books": 114,
        },
    },
    "common_rows_by_rule": {
        rule: int(
            all_flow.loc[
                rule,
                "common_support_contract_rows",
            ]
        )
        for rule in RULES
    },
    "complete_books_by_rule": {
        rule: int(
            all_flow.loc[
                rule,
                "complete_common_books",
            ]
        )
        for rule in RULES
    },
    "probability_bridge_retained": False,
    "artificial_ensemble_features_created": False,
    "final_modelling_split_assigned": False,
    "outcome_availability_history_applied": False,
    "modelling_readiness": (
        "Core empirical sample frozen. Historical HKO "
        "outcome-publication admissibility and the final "
        "chronological modelling split remain pending."
    ),
    "integrity_checks_passed": int(
        integrity["passed"].sum()
    ),
    "integrity_checks_total": int(
        len(integrity)
    ),
    "issue_rows": 0,
}

summary_path = OUT / "18s_expanded_sample_summary.json"
summary_path.write_text(
    json.dumps(
        summary,
        indent=2,
        ensure_ascii=False,
    ),
    encoding="utf-8",
)

environment = {
    "generated_at_utc": datetime.now(UTC).isoformat(),
    "python": sys.version,
    "platform": platform.platform(),
    "numpy": np.__version__,
    "pandas": pd.__version__,
    "timezone": "Asia/Hong_Kong",
    "decision_rules": RULES,
    "adapter_stage": "18sA",
    "freeze_stage": "18sB",
    "probability_bridge_retained": False,
    "final_modelling_split_assigned": False,
    "outcome_availability_history_applied": False,
}

environment_path = OUT / "18s_expanded_environment.json"
environment_path.write_text(
    json.dumps(environment, indent=2),
    encoding="utf-8",
)

all_binary = binary.loc[
    binary["sample_block"].eq("ALL")
].sort_values("decision_rule_order")
all_categorical = categorical.loc[
    categorical["sample_block"].eq("ALL")
].sort_values("decision_rule_order")
all_weather = weather_summary.loc[
    weather_summary["sample_block"].eq("ALL")
].sort_values("decision_rule_order")

report_lines = [
    "# 18s expanded March–June canonical sample freeze",
    "",
    "**PASS**",
    "",
    "## Pipeline structure",
    "",
    (
        "The release is divided into an 18sA schema-adapter "
        "stage and an 18sB aggregation-and-freeze stage."
    ),
    "",
    "## Frozen empirical universe",
    "",
    f"- Certified contracts: {len(target):,}",
    f"- Certified dates: {target['event_date'].nunique():,}",
    f"- Contract-decision candidates: {len(candidate):,}",
    f"- Market-ready rows: {len(market):,}",
    f"- Weather-ready rows: {len(weather):,}",
    f"- Exact common-support rows: {len(common):,}",
    (
        "- Common-support date-rule groups: "
        f"{len(common_group_sizes):,}"
    ),
    f"- Complete common-support books: {len(books):,}",
    (
        "- Complete-book contract rows: "
        f"{len(complete_contract):,}"
    ),
    f"- Explicit support exclusions: {len(exclusions):,}",
    "",
    "## Sample blocks",
    "",
    "| Block | Contracts | Dates | Common rows | Complete books |",
    "|---|---:|---:|---:|---:|",
    "| March-May baseline | 803 | 73 | 2,635 | 236 |",
    "| June external extension | 330 | 30 | 1,254 | 114 |",
    "| **Total** | **1,133** | **103** | **3,889** | **350** |",
    "",
    "## Support by decision rule",
    "",
    (
        "| Rule | Candidate | Market ready | Weather ready | "
        "Common rows | Common groups | Complete books |"
    ),
    "|---|---:|---:|---:|---:|---:|---:|",
]

for rule in RULES:
    row = all_flow.loc[rule]
    report_lines.append(
        "| {rule} | {candidate} | {market} | "
        "{weather} | {common} | {groups} | {books} |".format(
            rule=rule,
            candidate=int(
                row["candidate_contract_rows"]
            ),
            market=int(
                row["market_price_ready_rows"]
            ),
            weather=int(
                row["weather_path_ready_rows"]
            ),
            common=int(
                row["common_support_contract_rows"]
            ),
            groups=int(
                row["common_support_groups"]
            ),
            books=int(
                row["complete_common_books"]
            ),
        )
    )

report_lines.extend(
    [
        "",
        "## Market binary scores on exact support",
        "",
        "| Rule | n | Mean Brier | Mean log score |",
        "|---|---:|---:|---:|",
    ]
)

for row in all_binary.itertuples(index=False):
    report_lines.append(
        f"| {row.decision_rule} | "
        f"{int(row.n_contract_rows)} | "
        f"{float(row.mean_market_brier):.8f} | "
        f"{float(row.mean_market_log_score):.8f} |"
    )

report_lines.extend(
    [
        "",
        "## Complete-book diagnostics",
        "",
        (
            "| Rule | Books | Normalised categorical log | "
            "Normalised multiclass Brier | Modal-set winner rate | "
            "Deterministic bin hit rate |"
        ),
        "|---|---:|---:|---:|---:|---:|",
    ]
)

for row in all_categorical.itertuples(index=False):
    report_lines.append(
        "| {rule} | {books} | {log:.6f} | "
        "{brier:.6f} | {modal:.6f} | {det:.6f} |".format(
            rule=row.decision_rule,
            books=int(row.n_complete_books),
            log=float(
                row.mean_normalised_categorical_log_score
            ),
            brier=float(
                row.mean_normalised_multiclass_brier
            ),
            modal=float(
                row.market_modal_contains_actual_winner_rate
            ),
            det=float(
                row.deterministic_exact_contract_hit_rate
            ),
        )
    )

report_lines.extend(
    [
        "",
        "## Methodological boundary",
        "",
        (
            "No Gaussian-bridge probability or artificial "
            "ensemble feature is retained. The final modelling "
            "split and historical HKO outcome-publication "
            "admissibility remain deliberately unassigned."
        ),
    ]
)

report_path = (
    REPORT
    / "18s_expanded_march_june_canonical_sample_report.md"
)
report_path.write_text(
    "\n".join(report_lines) + "\n",
    encoding="utf-8",
)

manifest_rows = []

for root in [OUT, REPORT]:
    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue
        if path.name == "18s_expanded_sha256_manifest.csv":
            continue

        manifest_rows.append(
            {
                "path": str(path.relative_to(ROOT)),
                "size_bytes": path.stat().st_size,
                "sha256": sha256_file(path),
            }
        )

manifest_path = OUT / "18s_expanded_sha256_manifest.csv"
pd.DataFrame(manifest_rows).to_csv(
    manifest_path,
    index=False,
)

print(json.dumps(summary, indent=2))
print()
print("Expanded support by rule:")
display(
    flow.loc[
        flow["sample_block"].eq("ALL")
    ].sort_values("decision_rule_order")
)
print()
print("18sB expanded sample freeze: PASS")

18sA adapter input verification: PASS
Canonical targets: 1,133
Canonical market rows: 4,186
Canonical weather rows: 4,125


{
  "step": "18s",
  "generated_at_utc": "2026-07-21T20:55:21.456133+00:00",
  "verdict": "PASS",
  "pipeline_structure": [
    "18sA_canonical_source_adapters",
    "18sB_expanded_sample_freeze"
  ],
  "certified_contracts": 1133,
  "certified_dates": 103,
  "contract_decision_candidates": 4532,
  "market_ready_rows": 4186,
  "weather_ready_rows": 4125,
  "exact_common_support_rows": 3889,
  "common_support_date_rule_groups": 355,
  "complete_common_support_books": 350,
  "complete_book_contract_rows": 3850,
  "support_exclusion_rows": 643,
  "support_exclusions_by_reason": {
    "WEATHER_PATH_NOT_READY": 297,
    "MARKET_PRICE_MISSING": 236,
    "MARKET_PRICE_MISSING;WEATHER_PATH_NOT_READY": 110
  },
  "sample_blocks": {
    "march_may_baseline": {
      "contracts": 803,
      "dates": 73,
      "common_rows": 2635,
      "complete_books": 236
    },
    "june_external_extension": {
      "contracts": 330,
      "dates": 30,
      "common_rows": 1254,
      "complete_books": 114
   

,sample_block,decision_rule,decision_rule_order,candidate_contract_rows,market_price_ready_rows,weather_path_ready_rows,common_support_contract_rows,support_exclusion_rows,common_support_groups,complete_common_books,incomplete_common_groups
8,ALL,24h_prior,0,1133,976,1056,921,212,85,82,3
9,ALL,12h_prior,1,1133,1055,1023,978,155,89,88,1
10,ALL,6h_prior,2,1133,1077,1001,967,166,88,87,1
11,ALL,event_day_open,3,1133,1078,1045,1023,110,93,93,0



18sB expanded sample freeze: PASS
